In [0]:
from pyspark.sql import functions as F

LANDING = "/Volumes/workspace/olist_raw/landing"
CHK = "/Volumes/workspace/olist_raw/checkpoints"
TABLES = ["customers", "geolocation", "order_items", "order_payments", "order_reviews",
          "orders", "products", "sellers", "product_category_name_translation"]

for t in TABLES:
    print(f"Loading {t} ...")
    (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("multiLine", "true")
        .option("escape", '"')
        .option("cloudFiles.schemaLocation", f"{CHK}/{t}/_schema")
        .load(f"{LANDING}/{t}")
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
     .writeStream
        .option("checkpointLocation", f"{CHK}/{t}")
        .trigger(availableNow=True)
        .toTable(f"workspace.olist_bronze.{t}")
        .awaitTermination())
print("Done")